In [ ]:
!pip install wfdb neurokit2 plotly -q

In [ ]:
# ============================================================
# 🧠 FIXED: ECG MULTI-DISEASE CLASSIFICATION
# ============================================================
# Key fixes:
# 1. Proper data preprocessing (no per-class normalization)
# 2. Simpler model architecture
# 3. Focus on Arrhythmia (real ECG with good patterns)
# ============================================================

import os
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, optimizers, callbacks
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import roc_auc_score, classification_report, confusion_matrix
from google.colab import drive
import wfdb
from tqdm import tqdm
import matplotlib.pyplot as plt

# Mount Drive
drive.mount('/content/drive')
drive_path = "/content/drive/MyDrive/ECG_MultiDisease"
os.makedirs(drive_path, exist_ok=True)

# ============================================================
# FIXED: Download Arrhythmia Data (This one works well!)
# ============================================================

def download_arrhythmia_fixed():
    """Download MIT-BIH Arrhythmia with proper preprocessing"""
    print("📥 Downloading MIT-BIH Arrhythmia Database...")

    records = ['100', '101', '103', '105', '106', '107', '108', '109',
               '111', '112', '113', '114', '115', '116', '117', '118', '119',
               '121', '122', '123', '124', '200', '201', '202', '203']

    X_list, y_list = [], []
    arrhythmia_symbols = ['V', 'E', 'A', 'J', 'S', 'F', 'a', 'e', 'j']

    for record in tqdm(records, desc="Processing records"):
        try:
            signals, fields = wfdb.rdsamp(record, pn_dir='mitdb')
            annotation = wfdb.rdann(record, extension='atr', pn_dir='mitdb')

            fs = fields['fs']
            segment_length = int(10 * fs)  # 10 seconds

            for i in range(0, len(signals) - segment_length, segment_length // 2):
                segment = signals[i:i+segment_length, 0]

                # Resample to fixed length WITHOUT normalization
                segment = np.interp(np.linspace(0, len(segment), 3600),
                                   np.arange(len(segment)), segment)

                # Label based on annotations in this window
                mask = (annotation.sample >= i) & (annotation.sample < i + segment_length)
                segment_symbols = [annotation.symbol[idx] for idx in range(len(annotation.sample)) if mask[idx]]
                has_arrhythmia = 1 if any(s in arrhythmia_symbols for s in segment_symbols) else 0

                X_list.append(segment)
                y_list.append(has_arrhythmia)

        except Exception as e:
            print(f"⚠️  Error {record}: {e}")
            continue

    return np.array(X_list), np.array(y_list)

# Check if we need to re-download
arrhythmia_path = os.path.join(drive_path, "arrhythmia_fixed.npy")
arrhythmia_labels_path = os.path.join(drive_path, "arrhythmia_labels_fixed.npy")

if os.path.exists(arrhythmia_path) and os.path.exists(arrhythmia_labels_path):
    print("✅ Loading existing Arrhythmia data...")
    X_arrhythmia = np.load(arrhythmia_path)
    y_arrhythmia = np.load(arrhythmia_labels_path)
else:
    print("📥 Downloading fresh Arrhythmia data...")
    X_arrhythmia, y_arrhythmia = download_arrhythmia_fixed()
    np.save(arrhythmia_path, X_arrhythmia)
    np.save(arrhythmia_labels_path, y_arrhythmia)

print(f"\n📊 Dataset: {len(X_arrhythmia)} samples")
print(f"   Positive (Arrhythmia): {np.sum(y_arrhythmia)} ({100*np.mean(y_arrhythmia):.1f}%)")
print(f"   Negative (Normal): {len(y_arrhythmia)-np.sum(y_arrhythmia)} ({100*(1-np.mean(y_arrhythmia)):.1f}%)")

# ============================================================
# Build Simple, Robust Model
# ============================================================

def build_ecg_model(input_shape):
    """Simple CNN optimized for ECG classification"""
    model = keras.Sequential([
        layers.Input(shape=input_shape),

        # Conv block 1
        layers.Conv1D(32, 7, padding='same'),
        layers.BatchNormalization(),
        layers.Activation('relu'),
        layers.MaxPooling1D(2),
        layers.Dropout(0.2),

        # Conv block 2
        layers.Conv1D(64, 5, padding='same'),
        layers.BatchNormalization(),
        layers.Activation('relu'),
        layers.MaxPooling1D(2),
        layers.Dropout(0.2),

        # Conv block 3
        layers.Conv1D(128, 3, padding='same'),
        layers.BatchNormalization(),
        layers.Activation('relu'),
        layers.MaxPooling1D(2),
        layers.Dropout(0.3),

        # Global pooling and classification
        layers.GlobalAveragePooling1D(),
        layers.Dense(64, activation='relu'),
        layers.Dropout(0.4),
        layers.Dense(1, activation='sigmoid')
    ], name='ECG_Arrhythmia_Classifier')

    return model

# ============================================================
# CRITICAL: Preprocess CORRECTLY (avoid data leakage)
# ============================================================

print("\n" + "="*80)
print("🔧 PREPROCESSING")
print("="*80)

# Step 1: Split FIRST, before any preprocessing
X_train, X_temp, y_train, y_temp = train_test_split(
    X_arrhythmia, y_arrhythmia,
    test_size=0.3,
    random_state=42,
    stratify=y_arrhythmia
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp,
    test_size=0.5,
    random_state=42,
    stratify=y_temp
)

print(f"✅ Split completed:")
print(f"   Train: {len(X_train)} ({100*np.mean(y_train):.1f}% positive)")
print(f"   Val:   {len(X_val)} ({100*np.mean(y_val):.1f}% positive)")
print(f"   Test:  {len(X_test)} ({100*np.mean(y_test):.1f}% positive)")

# Step 2: Fit scaler ONLY on training data
print(f"\n⚖️  Scaling data (fit on train only)...")
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

# Step 3: Reshape for CNN
X_train_scaled = np.expand_dims(X_train_scaled, -1)
X_val_scaled = np.expand_dims(X_val_scaled, -1)
X_test_scaled = np.expand_dims(X_test_scaled, -1)

print(f"✅ Final shapes: Train={X_train_scaled.shape}, Val={X_val_scaled.shape}, Test={X_test_scaled.shape}")

# ============================================================
# Train Model
# ============================================================

print("\n" + "="*80)
print("🚀 TRAINING ARRHYTHMIA CLASSIFICATION MODEL")
print("="*80)

# Build and compile
model = build_ecg_model((3600, 1))

model.compile(
    optimizer=optimizers.Adam(learning_rate=5e-4),
    loss='binary_crossentropy',
    metrics=[keras.metrics.AUC(name='auc'), 'accuracy']
)

print(f"\n📊 Model Summary:")
model.summary()

# Compute class weights
cw = compute_class_weight('balanced', classes=np.unique(y_train), y=y_train)
class_weights = {int(c): float(w) for c, w in zip(np.unique(y_train), cw)}
print(f"\n⚖️  Class weights: {class_weights}")

# Train
print(f"\n🏋️ Training...")
history = model.fit(
    X_train_scaled, y_train,
    validation_data=(X_val_scaled, y_val),
    class_weight=class_weights,
    epochs=50,
    batch_size=64,
    callbacks=[
        callbacks.EarlyStopping(
            monitor='val_auc',
            patience=10,
            restore_best_weights=True,
            mode='max',
            verbose=1
        ),
        callbacks.ReduceLROnPlateau(
            monitor='val_loss',
            factor=0.5,
            patience=5,
            verbose=1
        )
    ],
    verbose=2
)

# ============================================================
# Evaluate
# ============================================================

print("\n" + "="*80)
print("📊 FINAL EVALUATION ON TEST SET")
print("="*80)

# Get predictions
y_pred = model.predict(X_test_scaled, verbose=0)
y_pred_binary = (y_pred > 0.5).astype(int).flatten()

# Calculate metrics
test_auc = roc_auc_score(y_test, y_pred)
cm = confusion_matrix(y_test, y_pred_binary)

print(f"\n✅ Test AUC: {test_auc:.4f}")

print(f"\nConfusion Matrix:")
print(f"                 Predicted")
print(f"                 Normal  Arrhythmia")
print(f"Actual Normal    {cm[0,0]:<7} {cm[0,1]:<7}")
print(f"       Arrhythmia{cm[1,0]:<7} {cm[1,1]:<7}")

print(f"\nDetailed Classification Report:")
print(classification_report(y_test, y_pred_binary,
                           target_names=['Normal', 'Arrhythmia'],
                           digits=4))

# ============================================================
# Visualize Training
# ============================================================

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Loss
axes[0].plot(history.history['loss'], label='Train Loss', linewidth=2)
axes[0].plot(history.history['val_loss'], label='Val Loss', linewidth=2)
axes[0].set_xlabel('Epoch', fontsize=12)
axes[0].set_ylabel('Loss', fontsize=12)
axes[0].set_title('Training vs Validation Loss', fontsize=14, fontweight='bold')
axes[0].legend(fontsize=10)
axes[0].grid(True, alpha=0.3)

# AUC
axes[1].plot(history.history['auc'], label='Train AUC', linewidth=2)
axes[1].plot(history.history['val_auc'], label='Val AUC', linewidth=2)
axes[1].set_xlabel('Epoch', fontsize=12)
axes[1].set_ylabel('AUC', fontsize=12)
axes[1].set_title('Training vs Validation AUC', fontsize=14, fontweight='bold')
axes[1].legend(fontsize=10)
axes[1].grid(True, alpha=0.3)
axes[1].axhline(y=0.5, color='r', linestyle='--', alpha=0.3, label='Random')

# Accuracy
axes[2].plot(history.history['accuracy'], label='Train Acc', linewidth=2)
axes[2].plot(history.history['val_accuracy'], label='Val Acc', linewidth=2)
axes[2].set_xlabel('Epoch', fontsize=12)
axes[2].set_ylabel('Accuracy', fontsize=12)
axes[2].set_title('Training vs Validation Accuracy', fontsize=14, fontweight='bold')
axes[2].legend(fontsize=10)
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Save model
model_save_path = os.path.join(drive_path, 'arrhythmia_model_final.keras')
model.save(model_save_path)
print(f"\n💾 Model saved to: {model_save_path}")

# ============================================================
# SAVE DATA FOR DASHBOARD (CRITICAL - NO NAME ERRORS!)
# ============================================================
print("\n💾 Saving test data and history for dashboard...")
np.savez(os.path.join(drive_path, 'test_data.npz'), 
         X_test=X_test_scaled, 
         y_test=y_test,
         y_pred=y_pred)
np.savez(os.path.join(drive_path, 'training_history.npz'), 
         history=history.history)
print("✅ Data saved successfully!")

# ============================================================
# Final Summary
# ============================================================

print("\n" + "="*80)
print("✅ TRAINING COMPLETE!")
print("="*80)
print(f"\n🎯 Final Results:")
print(f"   Test AUC: {test_auc:.4f}")
print(f"   Sensitivity (Recall): {cm[1,1]/(cm[1,1]+cm[1,0]):.4f}")
print(f"   Specificity: {cm[0,0]/(cm[0,0]+cm[0,1]):.4f}")

if test_auc > 0.90:
    print(f"\n✅ Excellent performance! AUC > 90%")
elif test_auc > 0.80:
    print(f"\n✅ Good performance! AUC > 80%")
elif test_auc > 0.70:
    print(f"\n⚠️  Moderate performance. AUC > 70%")
else:
    print(f"\n⚠️  Performance needs improvement. AUC < 70%")

print(f"\n💡 Notes:")
print(f"   - Apnea dataset skipped (data leakage issue)")
print(f"   - MI dataset skipped (download failed)")
print(f"   - Arrhythmia has real ECG patterns and should work well")
print(f"   - Expected AUC for Arrhythmia: 85-92%")


In [ ]:
# ============================================================
# 📊 SLEEK INTERACTIVE DASHBOARD - NO NAME ERRORS!
# ============================================================

import plotly.graph_objects as go
from plotly.subplots import make_subplots
import pandas as pd
import numpy as np
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, roc_curve
from tensorflow import keras
import os

print("🎨 Generating Sleek ECG Arrhythmia Dashboard...")
print("="*70)

# ============================================================
# LOAD ALL REQUIRED DATA (NO NAME ERRORS!)
# ============================================================

drive_path = "/content/drive/MyDrive/ECG_MultiDisease"
model_path = os.path.join(drive_path, 'arrhythmia_model_final.keras')
test_data_path = os.path.join(drive_path, "test_data.npz")
history_path = os.path.join(drive_path, "training_history.npz")

# Validate all files exist
if not os.path.exists(model_path):
    print("❌ ERROR: Model file not found!")
    print("💡 Please run the training cell first.")
    raise FileNotFoundError(f"Model not found at: {model_path}")

if not os.path.exists(test_data_path):
    print("❌ ERROR: Test data not found!")
    print("💡 Please run the training cell first (it now saves test data automatically).")
    raise FileNotFoundError(f"Test data not found at: {test_data_path}")

# Load everything
print("📂 Loading model and data...")
model = keras.models.load_model(model_path)
test_data = np.load(test_data_path)
X_test_scaled = test_data['X_test']
y_test = test_data['y_test']
y_pred = test_data['y_pred']

# Load history if available
history_dict = None
if os.path.exists(history_path):
    history_data = np.load(history_path, allow_pickle=True)
    history_dict = history_data['history'].item()
    print("✅ Training history loaded!")
else:
    print("⚠️  Training history not found - some charts will be skipped.")

# ============================================================
# CALCULATE ALL METRICS
# ============================================================

y_pred_binary = (y_pred > 0.5).astype(int).flatten()
test_auc = roc_auc_score(y_test, y_pred)
cm = confusion_matrix(y_test, y_pred_binary)
fpr, tpr, thresholds = roc_curve(y_test, y_pred)

# Classification report
report_dict = classification_report(
    y_test, y_pred_binary,
    target_names=['Normal', 'Arrhythmia'],
    output_dict=True
)

# Extract key metrics
sensitivity = report_dict['Arrhythmia']['recall']
specificity = report_dict['Normal']['recall']
accuracy = report_dict['accuracy']
precision = report_dict['Arrhythmia']['precision']
f1_score = report_dict['Arrhythmia']['f1-score']

print(f"✅ Metrics calculated: AUC={test_auc:.4f}, Sensitivity={sensitivity:.4f}")

# ============================================================
# DASHBOARD 1: MAIN PERFORMANCE OVERVIEW
# ============================================================

# Create sophisticated multi-panel dashboard
fig1 = make_subplots(
    rows=3, cols=3,
    specs=[
        [{"type": "indicator"}, {"type": "indicator"}, {"type": "indicator"}],
        [{"type": "xy", "colspan": 2}, None, {"type": "xy"}],
        [{"type": "heatmap", "colspan": 2}, None, {"type": "bar"}]
    ],
    subplot_titles=(
        "Test AUC Score", "Sensitivity (Recall)", "Specificity",
        "ROC Curve", "Prediction Distribution",
        "Confusion Matrix", "Performance Metrics"
    ),
    vertical_spacing=0.12,
    horizontal_spacing=0.10,
    row_heights=[0.25, 0.35, 0.40]
)

# --- Row 1: Key Metric Indicators ---

# AUC Indicator
fig1.add_trace(go.Indicator(
    mode="gauge+number+delta",
    value=test_auc,
    title={'text': "<b>Test AUC</b>", 'font': {'size': 16}},
    delta={'reference': 0.85, 'increasing': {'color': "green"}, 'decreasing': {'color': "red"}},
    gauge={
        'axis': {'range': [0.5, 1], 'tickwidth': 2},
        'bar': {'color': "#1f77b4", 'thickness': 0.8},
        'steps': [
            {'range': [0.5, 0.7], 'color': "#ffcccc"},
            {'range': [0.7, 0.85], 'color': "#fff9cc"},
            {'range': [0.85, 1], 'color': "#ccffcc"}
        ],
        'threshold': {
            'line': {'color': "darkgreen", 'width': 3},
            'thickness': 0.75,
            'value': 0.90
        }
    },
    number={'font': {'size': 40}},
    domain={'x': [0, 1], 'y': [0, 1]}
), row=1, col=1)

# Sensitivity Indicator
fig1.add_trace(go.Indicator(
    mode="number+delta",
    value=sensitivity,
    title={'text': "<b>Sensitivity</b><br><span style='font-size:11px;color:gray'>True Arrhythmia Detection</span>"},
    number={'font': {'size': 45, 'color': '#2ca02c'}, 'valueformat': '.3f'},
    delta={'reference': 0.80, 'increasing': {'color': "green"}, 'valueformat': '.3f'},
    domain={'x': [0, 1], 'y': [0, 1]}
), row=1, col=2)

# Specificity Indicator
fig1.add_trace(go.Indicator(
    mode="number+delta",
    value=specificity,
    title={'text': "<b>Specificity</b><br><span style='font-size:11px;color:gray'>True Normal Detection</span>"},
    number={'font': {'size': 45, 'color': '#ff7f0e'}, 'valueformat': '.3f'},
    delta={'reference': 0.85, 'increasing': {'color': "green"}, 'valueformat': '.3f'},
    domain={'x': [0, 1], 'y': [0, 1]}
), row=1, col=3)

# --- Row 2: ROC Curve and Prediction Distribution ---

# ROC Curve
fig1.add_trace(go.Scatter(
    x=fpr, y=tpr,
    mode='lines',
    name='ROC Curve',
    line=dict(color='#1f77b4', width=3),
    fill='tozeroy',
    fillcolor='rgba(31, 119, 180, 0.2)',
    hovertemplate='FPR: %{x:.3f}<br>TPR: %{y:.3f}<extra></extra>'
), row=2, col=1)

fig1.add_trace(go.Scatter(
    x=[0, 1], y=[0, 1],
    mode='lines',
    name='Random Classifier',
    line=dict(color='red', width=2, dash='dash'),
    hovertemplate='Random Guess<extra></extra>'
), row=2, col=1)

fig1.add_annotation(
    x=0.6, y=0.3,
    text=f"<b>AUC = {test_auc:.4f}</b>",
    showarrow=False,
    font=dict(size=16, color='darkblue'),
    bgcolor='rgba(255,255,255,0.8)',
    bordercolor='darkblue',
    borderwidth=2,
    row=2, col=1
)

fig1.update_xaxes(title_text="False Positive Rate", row=2, col=1, gridcolor='lightgray')
fig1.update_yaxes(title_text="True Positive Rate", row=2, col=1, gridcolor='lightgray')

# Prediction Distribution
fig1.add_trace(go.Histogram(
    x=y_pred[y_test == 0].flatten(),
    name='Normal',
    marker_color='#2ca02c',
    opacity=0.7,
    nbinsx=30,
    hovertemplate='Probability: %{x:.3f}<br>Count: %{y}<extra>Normal</extra>'
), row=2, col=3)

fig1.add_trace(go.Histogram(
    x=y_pred[y_test == 1].flatten(),
    name='Arrhythmia',
    marker_color='#d62728',
    opacity=0.7,
    nbinsx=30,
    hovertemplate='Probability: %{x:.3f}<br>Count: %{y}<extra>Arrhythmia</extra>'
), row=2, col=3)

# Add threshold annotation
fig1.add_annotation(
    x=0.5, y=1.0,
    text="Threshold",
    showarrow=False,
    xref='x3', yref='paper',
    xanchor='center',
    yanchor='bottom',
    font=dict(size=10, color='black')
)

fig1.update_xaxes(title_text="Predicted Probability", row=2, col=3, gridcolor='lightgray')
fig1.update_yaxes(title_text="Count", row=2, col=3, gridcolor='lightgray')

# --- Row 3: Confusion Matrix and Metrics Bar Chart ---

# Confusion Matrix Heatmap
z = cm[::-1]  # Flip for correct orientation
x_labels = ['Normal', 'Arrhythmia']
y_labels = ['Arrhythmia', 'Normal']

# Calculate percentages
cm_percent = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis] * 100

# Create annotations
annotations = []
for i, row in enumerate(z):
    for j, value in enumerate(row):
        percent = cm_percent[::-1][i, j]
        annotations.append(
            dict(
                x=x_labels[j], y=y_labels[i],
                text=f"<b>{value}</b><br>{percent:.1f}%",
                showarrow=False,
                font=dict(
                    color="white" if value > np.max(cm)/2 else "black",
                    size=16
                )
            )
        )

fig1.add_trace(go.Heatmap(
    z=z,
    x=x_labels,
    y=y_labels,
    colorscale='Blues',
    showscale=True,
    colorbar=dict(title="Count", len=0.4, y=0.15),
    hovertemplate='Predicted: %{x}<br>Actual: %{y}<br>Count: %{z}<extra></extra>'
), row=3, col=1)

for ann in annotations:
    fig1.add_annotation(ann, row=3, col=1)

fig1.update_xaxes(title_text="Predicted Label", row=3, col=1)
fig1.update_yaxes(title_text="True Label", row=3, col=1)

# Performance Metrics Bar Chart
metrics_names = ['Accuracy', 'Precision', 'Recall', 'F1-Score', 'Specificity']
metrics_values = [accuracy, precision, sensitivity, f1_score, specificity]
colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd']

fig1.add_trace(go.Bar(
    x=metrics_names,
    y=metrics_values,
    marker_color=colors,
    text=[f'{v:.3f}' for v in metrics_values],
    textposition='outside',
    textfont=dict(size=12, color='black'),
    hovertemplate='%{x}: %{y:.4f}<extra></extra>'
), row=3, col=3)

# Add target line as a horizontal line trace
fig1.add_trace(go.Scatter(
    x=metrics_names,
    y=[0.80] * len(metrics_names),
    mode='lines',
    line=dict(color='green', width=2, dash='dot'),
    name='Target (0.80)',
    showlegend=True,
    hovertemplate='Target: 0.80<extra></extra>'
), row=3, col=3)

fig1.update_xaxes(row=3, col=3)
fig1.update_yaxes(title_text="Score", range=[0, 1.05], row=3, col=3, gridcolor='lightgray')

# Overall Layout
fig1.update_layout(
    title_text="<b>🩺 ECG Arrhythmia Classification - Performance Dashboard</b>",
    title_x=0.5,
    title_font=dict(size=24, color='darkblue'),
    height=1200,
    showlegend=True,
    template='plotly_white',
    font=dict(family="Arial, sans-serif", size=12),
    legend=dict(
        orientation="h",
        yanchor="bottom", y=-0.05,
        xanchor="center", x=0.5
    ),
    hovermode='closest'
)

fig1.show()

# ============================================================
# DASHBOARD 2: TRAINING HISTORY (IF AVAILABLE)
# ============================================================

if history_dict is not None:
    epochs = list(range(1, len(history_dict['auc']) + 1))
    
    fig2 = make_subplots(
        rows=2, cols=2,
        subplot_titles=(
            'Training & Validation Loss',
            'Training & Validation AUC',
            'Training & Validation Accuracy',
            'Learning Progress Summary'
        ),
        specs=[
            [{"type": "xy"}, {"type": "xy"}],
            [{"type": "xy"}, {"type": "indicator"}]
        ],
        vertical_spacing=0.15,
        horizontal_spacing=0.12
    )
    
    # Loss curves
    fig2.add_trace(go.Scatter(
        x=epochs, y=history_dict['loss'],
        name='Train Loss',
        mode='lines+markers',
        line=dict(color='#1f77b4', width=2.5),
        marker=dict(size=4)
    ), row=1, col=1)
    
    fig2.add_trace(go.Scatter(
        x=epochs, y=history_dict['val_loss'],
        name='Validation Loss',
        mode='lines+markers',
        line=dict(color='#ff7f0e', width=2.5, dash='dash'),
        marker=dict(size=4)
    ), row=1, col=1)
    
    fig2.update_xaxes(title_text="Epoch", row=1, col=1, gridcolor='lightgray')
    fig2.update_yaxes(title_text="Loss", row=1, col=1, gridcolor='lightgray')
    
    # AUC curves
    fig2.add_trace(go.Scatter(
        x=epochs, y=history_dict['auc'],
        name='Train AUC',
        mode='lines+markers',
        line=dict(color='#2ca02c', width=2.5),
        marker=dict(size=4)
    ), row=1, col=2)
    
    fig2.add_trace(go.Scatter(
        x=epochs, y=history_dict['val_auc'],
        name='Validation AUC',
        mode='lines+markers',
        line=dict(color='#d62728', width=2.5, dash='dash'),
        marker=dict(size=4)
    ), row=1, col=2)
    
    # Add random baseline as a horizontal line
    fig2.add_trace(go.Scatter(
        x=epochs,
        y=[0.5] * len(epochs),
        mode='lines',
        line=dict(color='red', width=2, dash='dot'),
        name='Random Baseline',
        showlegend=True,
        hovertemplate='Random: 0.50<extra></extra>'
    ), row=1, col=2)
    
    fig2.update_xaxes(title_text="Epoch", row=1, col=2, gridcolor='lightgray')
    fig2.update_yaxes(title_text="AUC", range=[0.4, 1.02], row=1, col=2, gridcolor='lightgray')
    
    # Accuracy curves
    fig2.add_trace(go.Scatter(
        x=epochs, y=history_dict['accuracy'],
        name='Train Accuracy',
        mode='lines+markers',
        line=dict(color='#9467bd', width=2.5),
        marker=dict(size=4)
    ), row=2, col=1)
    
    fig2.add_trace(go.Scatter(
        x=epochs, y=history_dict['val_accuracy'],
        name='Validation Accuracy',
        mode='lines+markers',
        line=dict(color='#8c564b', width=2.5, dash='dash'),
        marker=dict(size=4)
    ), row=2, col=1)
    
    fig2.update_xaxes(title_text="Epoch", row=2, col=1, gridcolor='lightgray')
    fig2.update_yaxes(title_text="Accuracy", row=2, col=1, gridcolor='lightgray')
    
    # Training summary indicator
    best_epoch = np.argmax(history_dict['val_auc']) + 1
    best_val_auc = np.max(history_dict['val_auc'])
    
    fig2.add_trace(go.Indicator(
        mode="number",
        value=best_epoch,
        title={'text': f"<b>Best Epoch</b><br><span style='font-size:12px'>Val AUC: {best_val_auc:.4f}</span>"},
        number={'font': {'size': 50, 'color': '#1f77b4'}},
        domain={'x': [0, 1], 'y': [0, 1]}
    ), row=2, col=2)
    
    fig2.update_layout(
        title_text="<b>📈 Training History & Learning Curves</b>",
        title_x=0.5,
        title_font=dict(size=22, color='darkblue'),
        height=800,
        showlegend=True,
        template='plotly_white',
        font=dict(family="Arial, sans-serif", size=12),
        legend=dict(
            orientation="v",
            yanchor="top", y=1,
            xanchor="left", x=1.02
        )
    )
    
    fig2.show()

# ============================================================
# DASHBOARD 3: DETAILED CLASSIFICATION REPORT TABLE
# ============================================================

report_df = pd.DataFrame(report_dict).transpose().round(4)
report_df.index.name = 'Class / Metric'
report_df.reset_index(inplace=True)

# Create styled table
fig3 = go.Figure(data=[go.Table(
    header=dict(
        values=[f'<b>{col}</b>' for col in report_df.columns],
        fill_color='#4A90E2',
        align='center',
        font=dict(size=14, color='white', family='Arial'),
        height=40
    ),
    cells=dict(
        values=[report_df[col] for col in report_df.columns],
        fill_color=[['#F0F8FF', '#E6F2FF']*len(report_df)],
        align='center',
        font=dict(size=13, family='Arial'),
        height=35,
        format=[None, '.4f', '.4f', '.4f', None]
    )
)])

fig3.update_layout(
    title_text="<b>📋 Detailed Classification Report</b>",
    title_x=0.5,
    title_font=dict(size=20, color='darkblue'),
    height=400,
    template='plotly_white'
)

fig3.show()

# ============================================================
# SUMMARY STATISTICS
# ============================================================

print("\n" + "="*70)
print("📊 COMPREHENSIVE MODEL PERFORMANCE SUMMARY")
print("="*70)
print(f"\n🎯 PRIMARY METRICS:")
print(f"   ├─ Test AUC:        {test_auc:.4f} ({test_auc*100:.2f}%)")
print(f"   ├─ Accuracy:        {accuracy:.4f} ({accuracy*100:.2f}%)")
print(f"   └─ F1-Score:        {f1_score:.4f}")
print(f"\n🏥 CLINICAL METRICS:")
print(f"   ├─ Sensitivity:     {sensitivity:.4f} ({sensitivity*100:.2f}%) - Arrhythmia Detection Rate")
print(f"   ├─ Specificity:     {specificity:.4f} ({specificity*100:.2f}%) - Normal Detection Rate")
print(f"   └─ Precision:       {precision:.4f} ({precision*100:.2f}%) - Positive Prediction Accuracy")
print(f"\n📉 CONFUSION MATRIX:")
print(f"   ├─ True Negatives:  {cm[0,0]} (Correctly identified Normal)")
print(f"   ├─ False Positives: {cm[0,1]} (Normal misclassified as Arrhythmia)")
print(f"   ├─ False Negatives: {cm[1,0]} (Arrhythmia misclassified as Normal)")
print(f"   └─ True Positives:  {cm[1,1]} (Correctly identified Arrhythmia)")

# Performance evaluation
print(f"\n🎖️  PERFORMANCE GRADE:")
if test_auc >= 0.95:
    grade = "OUTSTANDING ⭐⭐⭐⭐⭐"
    comment = "Research-grade performance! Exceptional model!"
elif test_auc >= 0.90:
    grade = "EXCELLENT ⭐⭐⭐⭐"
    comment = "Very strong performance! Clinical potential!"
elif test_auc >= 0.85:
    grade = "VERY GOOD ⭐⭐⭐"
    comment = "Good performance! Reliable predictions!"
elif test_auc >= 0.80:
    grade = "GOOD ⭐⭐"
    comment = "Decent performance. Some room for improvement."
elif test_auc >= 0.75:
    grade = "FAIR ⭐"
    comment = "Functional but needs improvement."
else:
    grade = "NEEDS IMPROVEMENT"
    comment = "Consider retraining or adjusting architecture."

print(f"   {grade}")
print(f"   {comment}")
print("="*70)

print("\n✅ Dashboard generation complete!")
print("💡 All dashboards are interactive - hover over elements for details!")
